BI-DIRECTIONAL LSTM REGRESSION MODEL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, Dropout

# 1. Load and Scale Data
df = pd.read_csv('NIFTY50.csv').dropna()
data = df['Adj Close'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# 2. Create Sequences (Look-back window of 60 days)
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, 60)

# 3. Time-Series Split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 4. Build Bidirectional LSTM Model
model = Sequential([
    Bidirectional(LSTM(units=50, return_sequences=True), input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    Bidirectional(LSTM(units=50)),
    Dropout(0.2),
    Dense(units=25),
    Dense(units=1) # Output layer for regression
])

model.compile(optimizer='adam', loss='mean_squared_error')

# 5. Train the Model
model.fit(X_train, y_train, batch_size=32, epochs=10)

# 6. Predict and Inverse Scale
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
actual_prices = scaler.inverse_transform(y_test)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 19s 111ms/step - loss: 0.0114
Epoch 2/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 97ms/step - loss: 5.2489e-04
Epoch 3/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 115ms/step - loss: 4.4172e-04
Epoch 4/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 115ms/step - loss: 3.3239e-04
Epoch 5/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 19s 101ms/step - loss: 3.1707e-04
Epoch 6/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 12s 115ms/step - loss: 2.7632e-04
Epoch 7/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 107ms/step - loss: 2.5336e-04
Epoch 8/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 22s 118ms/step - loss: 2.4801e-04
Epoch 9/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 19s 105ms/step - loss: 2.1502e-04
Epoch 10/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 22s 115ms/step - loss: 2.1395e-04
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step


PERFORMANCE METRICS

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
def print_performance_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"Mean Absolute Error (MAE):      {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared Score (R2):          {r2:.4f}")
print_performance_metrics(actual_prices, predictions)

Mean Absolute Error (MAE):      205.13
Root Mean Squared Error (RMSE): 270.89
R-squared Score (R2):          0.9855
